# 임상 ML 학습

`model_choice.yaml`에서 모델 목록과 전처리를, 각 모델 `parameter.yaml`에서 하이퍼파라미터·early stopping·threshold를 설정합니다. 학습은 환자 ID로 고정 split을 적용하고 train에만 전처리를 fit합니다. test는 이 노트북에서 자동 평가하지 않습니다.

In [ ]:
import json
from pathlib import Path

import pandas as pd

repo = Path.cwd().resolve()
while not (repo / "Members/sang/model/ML/model_choice.yaml").is_file():
    if repo.parent == repo:
        raise RuntimeError("저장소 루트에서 노트북을 실행하세요")
    repo = repo.parent
choice_path = repo / "Members/sang/model/ML/model_choice.yaml"
from Members.sang.model.ML.training import train_clinical  # noqa: E402

## 학습 실행

아래 셀은 `model_choice.yaml`에 선택된 XGBoost, CatBoost, Random Forest를 학습합니다. XGBoost와 CatBoost는 validation Average Precision으로 best iteration을 고르고, Random Forest는 설정된 estimator 수로 학습한 뒤 validation AP를 기록합니다. 기능 확인용 소규모 실행은 `train_clinical(choice_path, limit_train_patients=32)`로 제한할 수 있습니다. 실제 실험은 인자를 생략합니다.

In [ ]:
run_dirs = train_clinical(choice_path)
run_dirs

## 결과 확인

각 run은 `ML/models/<run_id>/`에 저장됩니다. 여기에는 best model, train-fitted preprocessing, resolved YAML, audit, validation 환자 확률, 지표가 있습니다. Test 평가는 `evaluation/evaluation.py`에서 별도로 실행합니다.

In [ ]:
for run_dir in run_dirs:
    print(run_dir.name)
    display(pd.read_csv(run_dir / "validation_predictions.csv").head())
    print(json.loads((run_dir / "metrics.json").read_text())["validation"])